# Part 0: A gentle introduction to tensors

This notebook is predominantly for anyone who has not previously worked with NumPy arrays or PyTorch tensors, but we recommend everyone skims through for a refresher. It demonstrates the notation and syntax used by the remaining labs.

Work through it carefully before Part 1 if terms such as *shape*, *axis*, *slice*, or *matrix multiplication* are unfamiliar. Try to predict the result of each code cell before running it.

## From Python values to tensors

A single number is a **scalar**. A sequence of numbers can represent a **vector**, while a rectangular list of lists can represent a **matrix**. Ordinary Python lists let us store these structures.

In [ ]:
scalar = 3.5
vector_as_list = [1.0, 2.0, 3.0]
matrix_as_lists = [
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
]

print(scalar)
print(vector_as_list)
print(matrix_as_lists)
print("Second row, third value:", matrix_as_lists[1][2])

Python lists are general-purpose containers, not mathematical arrays. Multiplying a list by two repeats it; adding lists concatenates them. Neither operation performs the element-wise arithmetic we usually want in machine learning.

In [ ]:
print(vector_as_list * 2)
print(vector_as_list + [4.0, 5.0])

A **tensor** is a regular, multidimensional array of values. In this course, a tensor is the basic data structure used to represent examples, model parameters, predictions, and gradients.

PyTorch tensors play a role similar to NumPy arrays, while also supporting accelerator devices and automatic differentiation. We can create a tensor from an existing Python list.

In [ ]:
import torch

vector = torch.tensor(vector_as_list)
matrix = torch.tensor(matrix_as_lists)

print(vector)
print(matrix)
print(type(matrix))
print(matrix.tolist())

Every tensor has several useful properties:

- `shape`: the size of each axis;
- `ndim`: the number of axes;
- `numel()`: the total number of values;
- `dtype`: the representation used for each value; and
- `device`: where the values are stored and computations occur.

In [ ]:
print("shape: ", matrix.shape)
print("axes:  ", matrix.ndim)
print("values:", matrix.numel())
print("dtype: ", matrix.dtype)
print("device:", matrix.device)

## Shapes and axes

The shape tells us how values are organised. A shape of `(2, 3)` means two rows and three columns. The total number of values is therefore $2\times3=6$.

The axes are numbered from zero:

- axis `0` of a matrix selects rows;
- axis `1` selects columns.

For a batch of data, axis `0` will usually select examples.

In [ ]:
scalar_tensor = torch.tensor(3.5)
vector_tensor = torch.tensor([1.0, 2.0, 3.0])
matrix_tensor = torch.zeros(2, 3)
batch_tensor = torch.zeros(5, 2, 3)

print("scalar:", scalar_tensor.shape)
print("vector:", vector_tensor.shape)
print("matrix:", matrix_tensor.shape)
print("batch: ", batch_tensor.shape)

## Indexing

Python and PyTorch use **zero-based indexing**: the first position is numbered `0`, the second is `1`, and so on. Negative indices count backwards: `-1` is the final position and `-2` is the penultimate position.

For a matrix, `matrix[row, column]` selects one value.

In [ ]:
numbers = torch.tensor([10, 20, 30, 40, 50])
grid = torch.tensor([
    [ 0,  1,  2,  3],
    [10, 11, 12, 13],
    [20, 21, 22, 23],
])

print(numbers[0], numbers[-1], numbers[-2])
print(grid[1, 2])
print(grid[-1, -1])
print(grid[1])

### Task: select by position

Using `grid`, select the top-left value, bottom-right value, and second row. Do not type the resulting values directly.

In [ ]:
# TODO: replace the None values with indexing expressions.
top_left = None
bottom_right = None
second_row = None

In [ ]:
assert top_left.item() == 0
assert bottom_right.item() == 23
torch.testing.assert_close(second_row, torch.tensor([10, 11, 12, 13]))
print("Task 1 checks passed")

## Slicing

A slice selects a range. The basic form is `start:stop`, where `start` is included and `stop` is excluded. This half-open convention means `x[1:4]` contains positions `1`, `2`, and `3`.

Either end may be omitted:

- `x[:3]`: from the beginning up to, but not including, position 3;
- `x[2:]`: from position 2 to the end;
- `x[:]`: everything;
- `x[::2]`: everything, taking every second value.

In [ ]:
print(numbers[1:4])
print(numbers[:3])
print(numbers[2:])
print(numbers[::2])
print(grid[0:2, 1:3])

Selecting one position removes an axis; slicing a range preserves it. Consequently, `grid[:, 0]` has shape `(3,)`, while `grid[:, 0:1]` has shape `(3, 1)`. Both contain the first column, but they represent it differently.

In [ ]:
column_vector = grid[:, 0]
column_matrix = grid[:, 0:1]
print(column_vector, column_vector.shape)
print(column_matrix, column_matrix.shape)

### Indexing with an ellipsis

An ellipsis, written `...`, stands for as many complete axes as are needed to make an indexing expression fit the tensor. Only one ellipsis is needed in an indexing expression. For a tensor with three axes:

- `x[..., -1]` is equivalent to `x[:, :, -1]`;
- `x[..., 1:3]` is equivalent to `x[:, :, 1:3]`; and
- `x[0, ...]` is equivalent to `x[0, :, :]`.

This is particularly convenient when we care about a trailing feature axis but do not want the code to depend on the number of leading batch-like axes. Predict the shapes printed below before running the cell.

In [ ]:
cube_for_indexing = torch.arange(24).reshape(2, 3, 4)
last_value_from_each_row = cube_for_indexing[..., -1]
first_item = cube_for_indexing[0, ...]

print(last_value_from_each_row.shape)
print(first_item.shape)
torch.testing.assert_close(
    cube_for_indexing[..., -1], cube_for_indexing[:, :, -1]
)

sequence_features = torch.zeros(8, 20, 64)  # (batch, tokens, features)
print(sequence_features[..., 0].shape)       # one feature
print(sequence_features[..., :4].shape)      # first four features

### Task: slice columns and blocks

First use `:` to select the first column of `grid`. Then create a `5 × 6` tensor containing the integers from 0 to 29 and use one slicing expression to select rows 1 through 3 and columns 2 through 4, producing a tensor of shape `(3, 3)`.

In [ ]:
large_grid = torch.arange(30).reshape(5, 6)
# TODO
first_column = None
block = None

torch.testing.assert_close(first_column, torch.tensor([0, 10, 20]))
assert block.shape == (3, 3)
torch.testing.assert_close(
    block, torch.tensor([[8, 9, 10], [14, 15, 16], [20, 21, 22]])
)
print("Task 2 checks passed")

## Common ways to create tensors

PyTorch provides factory functions for common initial values. Their arguments usually describe the desired shape.

In [ ]:
torch.manual_seed(42)

print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.arange(0, 10, 2))
print(torch.linspace(0, 1, 5))
print(torch.randn(2, 3))

Tensors are normally homogeneous: every value has the same dtype. Floating-point tensors are used for model inputs, parameters, and most calculations. Integer tensors commonly store class labels or indices. Boolean tensors store masks.

In [ ]:
measurements = torch.tensor([1.2, 3.4], dtype=torch.float32)
class_labels = torch.tensor([0, 2], dtype=torch.int64)
mask = torch.tensor([True, False], dtype=torch.bool)

print(measurements.dtype, class_labels.dtype, mask.dtype)

## Element-wise arithmetic

For tensors of compatible shapes, `+`, `-`, `*`, `/`, and `**` operate element by element. In particular, `*` is element-wise multiplication rather than matrix multiplication. A scalar can also be applied to every element.

In [ ]:
a = torch.tensor([2.0, 4.0, 8.0])
b = torch.tensor([1.0, 2.0, 4.0])

print("a + b: ", a + b)
print("a - b: ", a - b)
print("a * b: ", a * b)
print("a / b: ", a / b)
print("a ** 2:", a ** 2)
print("a * 10:", a * 10)

### Task: combine element-wise operations

Without a loop, compute the midpoint `(p + q) / 2` and the squared element-wise difference `(p - q) ** 2`.

In [ ]:
p = torch.tensor([1.0, 4.0, 7.0])
q = torch.tensor([3.0, 2.0, 5.0])
# TODO
midpoint = None
squared_difference = None

torch.testing.assert_close(midpoint, torch.tensor([2.0, 3.0, 6.0]))
torch.testing.assert_close(squared_difference, torch.tensor([4.0, 4.0, 4.0]))
print("Task 3 checks passed")

## Matrix multiplication with `@`

Matrix multiplication combines rows from the left matrix with columns from the right matrix. It uses the `@` operator.

If `left` has shape `(m, k)` and `right` has shape `(k, n)`, then `left @ right` has shape `(m, n)`. The two inner sizes must agree.

In [ ]:
left = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # (2, 3)
right = torch.tensor([[1.0, 2.0], [0.0, 1.0], [2.0, 0.0]])  # (3, 2)

product = left @ right                                      # (2, 2)
print(product)
print(product.shape)

first_entry_by_hand = 1.0 * 1.0 + 2.0 * 0.0 + 3.0 * 2.0
print(product[0, 0], first_entry_by_hand)

### Task: check the shapes

Create `matmul_result` using one `@` operation. Predict its shape before running the checks.

In [ ]:
x_matrix = torch.arange(12, dtype=torch.float32).reshape(3, 4)
weight_matrix = torch.ones(4, 2)
# TODO
matmul_result = None

assert matmul_result.shape == (3, 2)
torch.testing.assert_close(matmul_result[0], torch.tensor([6.0, 6.0]))
print("Task 4 checks passed")

## Reshaping tensors

Reshaping changes how the same number of values is organised into axes. The total number of values must remain unchanged. A dimension written as `-1` asks PyTorch to infer its size.

In [ ]:
values = torch.arange(12)
as_three_by_four = values.reshape(3, 4)
as_two_by_six = values.reshape(2, -1)
back_to_vector = as_three_by_four.reshape(-1)

print(values)
print(as_three_by_four)
print(as_two_by_six.shape)
print(back_to_vector.shape)

### `view` and `reshape`

`view` returns another view of the same underlying storage, so modifying the view modifies the original tensor. It can only represent shapes compatible with the tensor's current memory layout.

`reshape` has the same convenient shape syntax, but it will create a copy when a view is not possible. It may therefore return either a view or a copy; code should not rely on which occurred.

In [ ]:
original = torch.arange(6)
viewed = original.view(2, 3)
viewed[0, 0] = -99
print(viewed)
print(original)  # the first value changed here too

original[0] = 0  # restore it for later examples

Some operations, such as transposing a matrix, change the logical order of axes without rearranging the underlying storage. Such a tensor is often **non-contiguous**. `view` may then fail, whereas `reshape` will produce the requested result.

In [ ]:
matrix_for_transpose = torch.arange(12).reshape(3, 4)
transposed = matrix_for_transpose.T
print("contiguous?", transposed.is_contiguous())

try:
    transposed.view(12)
except RuntimeError as error:
    print("view failed because the memory layout is incompatible")

flattened = transposed.reshape(12)
print(flattened)

A reasonable default is to use `reshape` when you simply need a shape and `view` when you specifically require a shared-storage view. Neither operation swaps the meaning of axes: use operations such as transpose or `permute` when axes must be reordered.

### Task: organise and restore values

Reshape the integers from 0 to 23 into a tensor with shape `(2, 3, 4)`, then restore it to a one-dimensional tensor. Use `-1` in one of the two operations.

In [ ]:
values_24 = torch.arange(24)
# TODO
cube = None
restored_24 = None

assert cube.shape == (2, 3, 4)
assert restored_24.shape == (24,)
torch.testing.assert_close(restored_24, values_24)
print("Task 5 checks passed")

## Conventions used in the course

Tensor shapes only become meaningful when we say what each axis represents. We will normally place the batch axis first and features last, except for image tensors where PyTorch conventionally places channels second. Common annotations include:

```python
tabular_data.shape     # (batch, features)
image_batch.shape      # (batch, channels, height, width)
token_embeddings.shape # (batch, tokens, embedding_features)
```

Use names such as `batch_size`, `num_features`, and `num_classes`, and write shape comments beside less obvious operations. This makes many tensor errors visible before running the code.

## Wrap-up challenge

Starting from `torch.arange(24, dtype=torch.float32)`: 

1. reshape it to `(2, 3, 4)`;
2. select the final row from every item in the first axis, producing `(2, 4)`;
3. multiply those values element-wise by `0.5`; and
4. multiply the result by a `(4, 2)` matrix of ones using `@`.

The final tensor should have shape `(2, 2)`.

In [ ]:
# TODO
source_values = torch.arange(24, dtype=torch.float32)
organised = None
final_rows = None
scaled_rows = None
final_result = None

assert organised.shape == (2, 3, 4)
assert final_rows.shape == (2, 4)
assert final_result.shape == (2, 2)
torch.testing.assert_close(final_result, torch.tensor([[19.0, 19.0], [43.0, 43.0]]))
print("Consolidation checks passed")

## Summary

You should now be able to read basic PyTorch tensor syntax and reason about shapes before executing code. Part 1 builds on this foundation by introducing reductions, devices, and the shape conventions used for batches of model inputs.